In [ ]:
# import pandas as pd
# import glob
# import os
# import re

# # ── Column rename map ─────────────────────────────────────────────────────
# # Original: food, size, access, income, poverty, last donation, total donation, dist,
# #           size.1, access.1, income.1, poverty.1, last donation.1, total donation.1, dist.1,
# #           num Q, AorB
# # food is shared between A and B (same food type in both options)
# # The remaining 7 attributes repeat for A and B

# COL_RENAME = {
#     "food":               "food",
#     "size":               "size_A",
#     "access":             "access_A",
#     "income":             "income_A",
#     "poverty":            "poverty_A",
#     "last donation":      "last_donation_A",
#     "total donation":     "total_donation_A",
#     "dist":               "dist_A",
#     "size.1":             "size_B",
#     "access.1":           "access_B",
#     "income.1":           "income_B",
#     "poverty.1":          "poverty_B",
#     "last donation.1":    "last_donation_B",
#     "total donation.1":   "total_donation_B",
#     "dist.1":             "dist_B",
#     "num Q":              "num_Q",
#     "AorB":               "AorB",
# }

# # ── Load and concatenate all CSVs ─────────────────────────────────────────
# RAW_DIR = "raw_data"
# pattern = os.path.join(RAW_DIR, "Copy of Allparticipants.xlsx - *.csv")

# frames = []
# for path in sorted(glob.glob(pattern)):
#     # Extract person ID from filename, e.g. '...F1.csv' -> 'F1'
#     person_id = re.search(r' - ([A-Z0-9]+)\.csv$', path).group(1)
#     df = pd.read_csv(path)
#     df = df.rename(columns=COL_RENAME)
#     df.insert(0, "personID", person_id)
#     frames.append(df)
#     print(f"  {person_id}: {len(df)} rows")

# combined = pd.concat(frames, ignore_index=True)
# print(f"\nCombined: {len(combined)} rows from {combined['personID'].nunique()} participants")
# print(f"Columns: {combined.columns.tolist()}")
# combined.head()

In [ ]:
# # ── Save to CSV ───────────────────────────────────────────────────────────
# OUT_PATH = "food_rescue_combined.csv"
# combined.to_csv(OUT_PATH, index=False)
# print(f"Saved to {OUT_PATH}")

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
import pandas as pd
import glob
import os
import re

combined = pd.read_csv('food_rescue_combined.csv')
# ── Feature columns (excluding food) ──────────────────────────────────────
FEAT_COLS = ["size", "access", "income", "poverty", "last_donation", "total_donation", "dist"]

def fit_btl_person(df_person, feat_cols=FEAT_COLS, C=1.0):
    """
    Fit a BTL model for one person.
    α = features_A − features_B
    y = 1 if chose A, 0 if chose B
    Returns θ (len(feat_cols),)
    """
    A_cols = [f"{f}_A" for f in feat_cols]
    B_cols = [f"{f}_B" for f in feat_cols]
    alpha = df_person[A_cols].values - df_person[B_cols].values
    y     = (df_person["AorB"] == "A").astype(int).values
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(fit_intercept=False, C=C, solver="lbfgs",
                            max_iter=2000, tol=1e-8)
    lr.fit(alpha, y)
    return lr.coef_.ravel()

# ── Fit per person ─────────────────────────────────────────────────────────
records = []
skipped = []
for person_id, grp in combined.groupby("personID"):
    theta = fit_btl_person(grp)
    if theta is None:
        skipped.append(person_id)
        continue
    records.append({"personID": person_id, **dict(zip(FEAT_COLS, theta))})

if skipped:
    print(f"Skipped (only one class): {skipped}")

theta_df = pd.DataFrame(records).set_index("personID")
print(f"Fit {len(theta_df)} participants x {len(FEAT_COLS)} features\n")

pd.set_option("display.float_format", "{:+.4f}".format)
pd.set_option("display.max_columns", None)
print(theta_df.to_string())
pd.reset_option("display.float_format")
theta_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# ── Manual clustering by stakeholder type (first letter of personID) ─────
theta_df["stakeholder"] = theta_df.index.str[0]
print("Manual stakeholder groups:")
print(theta_df["stakeholder"].value_counts().sort_index().to_string())

fig, ax = plt.subplots(figsize=(10, 4))
manual_centroids = theta_df.groupby("stakeholder")[FEAT_COLS].mean()
sns.heatmap(manual_centroids, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.5, ax=ax)
ax.set_title("Manual Stakeholder Group Centroids", fontweight="bold")
ax.set_xlabel("Feature")
ax.set_ylabel("Stakeholder type")
plt.tight_layout()
plt.show()

# ── K-means: sweep over k ─────────────────────────────────────────────────
X = theta_df[FEAT_COLS].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

K_RANGE = range(2, min(len(theta_df), 10))
inertias, silhouettes = [], []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=50, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_RANGE), inertias, marker="o", color="steelblue")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia (within-cluster SS)")
axes[0].set_title("Elbow plot", fontweight="bold")
axes[0].set_xticks(list(K_RANGE))

axes[1].plot(list(K_RANGE), silhouettes, marker="o", color="tomato")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette score (higher = better)", fontweight="bold")
axes[1].set_xticks(list(K_RANGE))

best_k = list(K_RANGE)[silhouettes.index(max(silhouettes))]
axes[1].axvline(best_k, color="k", linestyle="--", linewidth=1,
                label=f"best k={best_k}")
axes[1].legend()
plt.suptitle("K-means sweep", fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Best k by silhouette: {best_k}")

# ── Fit final k-means at best_k ───────────────────────────────────────────
km_final = KMeans(n_clusters=best_k, n_init=50, random_state=42)
theta_df["kmeans_cluster"] = km_final.fit_predict(X_scaled)

print(f"\nK-means cluster sizes (k={best_k}):")
print(theta_df["kmeans_cluster"].value_counts().sort_index().to_string())

# ── Group makeup: stakeholder composition of each k-means cluster ─────────
print("\nStakeholder composition of each k-means cluster:")
makeup = (theta_df.groupby(["kmeans_cluster", "stakeholder"])
          .size()
          .unstack(fill_value=0))
makeup["total"] = makeup.sum(axis=1)
print(makeup.to_string())

print("\nAs percentages:")
pct = makeup.drop(columns="total").div(makeup["total"], axis=0).mul(100).round(1)
print(pct.to_string())

# ── K-means centroid heatmap ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, max(3, best_k * 0.7)))
km_centroids = theta_df.groupby("kmeans_cluster")[FEAT_COLS].mean()
sns.heatmap(km_centroids, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.5, ax=ax)
ax.set_title(f"K-means Cluster Centroids (k={best_k})", fontweight="bold")
ax.set_xlabel("Feature")
ax.set_ylabel("Cluster")
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution of individual θ vectors ─────────────────────────────────
SK_COLORS = {'D': '#e07b54', 'F': '#5b8db8', 'R': '#6abf69', 'V': '#9b6ebf'}
SK_LABELS = {'D': 'Donor', 'F': 'Food bank', 'R': 'Recipient', 'V': 'Volunteer'}

theta_plot = theta_df[FEAT_COLS].copy()

fig, ax = plt.subplots(figsize=(10, 5))
rng_j = np.random.default_rng(0)
for i, feat in enumerate(FEAT_COLS):
    for pid, val in zip(theta_plot.index, theta_plot[feat]):
        jitter = rng_j.uniform(-0.22, 0.22)
        ax.scatter(i + jitter, val, color=SK_COLORS[pid[0]], s=55, alpha=0.85, zorder=3)
    mean_val = theta_plot[feat].mean()
    ax.plot([i - 0.35, i + 0.35], [mean_val, mean_val], color='black', lw=1.8, zorder=4)

ax.axhline(0, color='grey', ls='--', lw=0.7)
ax.set_xticks(range(len(FEAT_COLS)))
ax.set_xticklabels(FEAT_COLS, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('θ value')
ax.set_title('Individual θ per feature  (black bar = mean)', fontweight='bold')

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=SK_COLORS[k], label=f'{k} — {SK_LABELS[k]}')
                  for k in sorted(SK_COLORS)]
ax.legend(handles=legend_handles, fontsize=9, loc='lower left')

plt.suptitle('Distribution of individual θ vectors — 412 Food Rescue (n=19)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print("Summary statistics per feature:")
print(theta_df[FEAT_COLS].describe().round(3).to_string())


In [ ]:
# ── Histogram of θ values per feature ────────────────────────────────────
fig, axes = plt.subplots(1, len(FEAT_COLS), figsize=(18, 3.5))

for ax, feat in zip(axes, FEAT_COLS):
    vals = theta_df[feat].values
    ax.hist(vals, bins=10, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.axvline(vals.mean(), color='crimson', lw=1.5, ls='--', label=f'mean={vals.mean():.2f}')
    ax.axvline(0, color='grey', lw=0.8, ls=':')
    ax.set_title(feat, fontweight='bold', fontsize=9)
    ax.set_xlabel('\u03b8 value', fontsize=8)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=7)

axes[0].set_ylabel('# participants')
plt.suptitle('Distribution of individual \u03b8 values per feature \u2014 412 Food Rescue (n=19)',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.special import expit as sigmoid

# ── Shared helpers ────────────────────────────────────────────────────────
def fit_btl(alpha, y=None, C=1.0):
    """Fit BTL via logistic regression on α vectors. y defaults to all-ones (α = chosen−unchosen)."""
    if y is None:
        y = np.ones(len(alpha))
    lr = LogisticRegression(fit_intercept=False, C=C, solver="lbfgs",
                            max_iter=2000, tol=1e-8)
    lr.fit(alpha, y)
    return lr.coef_.ravel()

def build_alpha_y(df, feat_cols=FEAT_COLS):
    """Build (alpha, y, person_index) arrays from combined dataframe."""
    A_cols = [f"{f}_A" for f in feat_cols]
    B_cols = [f"{f}_B" for f in feat_cols]
    alpha = df[A_cols].values - df[B_cols].values
    y     = (df["AorB"] == "A").astype(int).values
    return alpha.astype(float), y

def compute_welfare_externalities(alpha_train, y_train, group_arr,
                                  theta_groups, theta_hat, weights,
                                  regime_name=""):
    """
    Compute per-group welfare and alignment externalities.
    q_dep = q_train (empirical, same distribution).

    W_g(θ) = E_z[ 2(α^T θ_g)(σ(α^T θ) − 0.5) ]
    θ̄_clust = Σ_g w_g θ_g
    η = J^{-1} · E_z[(α^T θ̄_clust) v(α^T θ̂) α]
    Γ_{g,z} = (σ(α^T θ_g) − σ(α^T θ̂)) · (α^T η)
    """
    K = len(theta_groups)
    alpha = alpha_train
    n = len(alpha)

    # θ̄_clust
    theta_bar_clust = sum(w * np.array(th, dtype=float) for w, th in zip(weights, theta_groups))

    # Deployment = training
    alpha_dep     = alpha
    sigma_hat_dep = sigmoid(alpha_dep @ theta_hat)
    v_hat_dep     = sigma_hat_dep * (1 - sigma_hat_dep)
    sigma_opt_dep = sigmoid(alpha_dep @ theta_bar_clust)

    # Welfare table
    results = []
    for g in range(K):
        logits_g = alpha_dep @ theta_groups[g]
        w_ideal  = 0.5 * np.abs(logits_g).mean()
        w_opt    = (logits_g * (sigma_opt_dep - 0.5)).mean()
        w_rlhf   = (logits_g * (sigma_hat_dep - 0.5)).mean()
        results.append({
            "group": g, "weight": weights[g],
            "W_ideal": w_ideal, "W_opt": w_opt, "W_rlhf": w_rlhf,
            "rlhf_ratio":  w_rlhf / w_opt   if abs(w_opt)   > 1e-10 else np.nan,
            "total_ratio": w_rlhf / w_ideal  if abs(w_ideal) > 1e-10 else np.nan,
        })
    welfare_df = pd.DataFrame(results)

    # Fisher info J at θ̂ (training)
    v_train = sigmoid(alpha @ theta_hat) * (1 - sigmoid(alpha @ theta_hat))
    J     = (alpha * v_train[:, None]).T @ alpha / n
    J_inv = np.linalg.inv(J + 1e-6 * np.eye(alpha.shape[1]))

    # η
    welfare_grad = (alpha_dep * ((alpha_dep @ theta_bar_clust) * v_hat_dep)[:, None]).mean(axis=0)
    eta = J_inv @ welfare_grad

    # Γ_{g,z}: (n_queries, K)
    adot = alpha_dep @ eta
    gamma_mat = np.column_stack([
        (sigmoid(alpha_dep @ theta_groups[g]) - sigma_hat_dep) * adot
        for g in range(K)
    ])

    print(f"\n{'='*60}")
    print(f"Regime: {regime_name}")
    print(f"{'='*60}")
    print(f"  ||θ̄_clust|| = {np.linalg.norm(theta_bar_clust):.4f}   "
          f"||θ̂|| = {np.linalg.norm(theta_hat):.4f}")
    print("\n── Welfare Table ──")
    pd.set_option("display.float_format", "{:+.4f}".format)
    print(welfare_df.round(4).to_string(index=False))
    pd.reset_option("display.float_format")

    return {"welfare_df": welfare_df, "eta": eta, "gamma_mat": gamma_mat,
            "alpha_dep": alpha_dep, "sigma_hat_dep": sigma_hat_dep,
            "theta_bar_clust": theta_bar_clust}

# ── Build shared alpha matrix and pooled θ̂ ───────────────────────────────
alpha_all, y_all = build_alpha_y(combined)
theta_hat = fit_btl(alpha_all, y_all)
print(f"Pooled θ̂ (all {len(alpha_all)} observations):")
print(pd.Series(theta_hat, index=FEAT_COLS).round(4).to_string())

# ══════════════════════════════════════════════════════════════════════════
# Regime 1: Individual (19 persons, one θ per person)
# ══════════════════════════════════════════════════════════════════════════
person_ids   = theta_df.index.tolist()
theta_indiv  = [theta_df.loc[pid, FEAT_COLS].values.astype(float) for pid in person_ids]

# Group array: assign each observation to its person index
pid_to_idx = {pid: i for i, pid in enumerate(person_ids)}
group_indiv = combined["personID"].map(pid_to_idx).values

# Weights: fraction of training observations per person
N = len(alpha_all)
weights_indiv = np.array([(group_indiv == i).sum() / N for i in range(len(person_ids))])

results_indiv = compute_welfare_externalities(
    alpha_all, y_all, group_indiv,
    theta_indiv, theta_hat, weights_indiv,
    regime_name=f"Individual ({len(person_ids)} persons)"
)

# ══════════════════════════════════════════════════════════════════════════
# Regime 2: Manual clustering (stakeholder type = first letter of personID)
# ══════════════════════════════════════════════════════════════════════════
stakeholder_types = sorted(theta_df["stakeholder"].unique())
stype_to_idx = {s: i for i, s in enumerate(stakeholder_types)}
theta_df["stakeholder_idx"] = theta_df["stakeholder"].map(stype_to_idx)
group_manual = combined["personID"].map(
    theta_df["stakeholder_idx"].to_dict()
).values.astype(int)

theta_manual = []
for s in stakeholder_types:
    mask = group_manual == stype_to_idx[s]
    theta_s = fit_btl(alpha_all[mask], y_all[mask])
    theta_manual.append(theta_s)
    print(f"  {s}: {mask.sum()} obs  ||θ||={np.linalg.norm(theta_s):.3f}")

weights_manual = np.array([(group_manual == i).sum() / N
                            for i in range(len(stakeholder_types))])

results_manual = compute_welfare_externalities(
    alpha_all, y_all, group_manual,
    theta_manual, theta_hat, weights_manual,
    regime_name=f"Manual clustering ({', '.join(stakeholder_types)})"
)
# label welfare rows
results_manual["welfare_df"]["group_label"] = stakeholder_types

# ══════════════════════════════════════════════════════════════════════════
# Regime 3: K-means clustering (best_k from sweep)
# ══════════════════════════════════════════════════════════════════════════
group_kmeans_series = theta_df["kmeans_cluster"]
group_kmeans = combined["personID"].map(group_kmeans_series.to_dict()).values.astype(int)

theta_kmeans = []
for g in range(best_k):
    mask = group_kmeans == g
    theta_g = fit_btl(alpha_all[mask], y_all[mask])
    theta_kmeans.append(theta_g)
    print(f"  cluster {g}: {mask.sum()} obs  ||θ||={np.linalg.norm(theta_g):.3f}")

weights_kmeans = np.array([(group_kmeans == g).sum() / N for g in range(best_k)])

results_kmeans = compute_welfare_externalities(
    alpha_all, y_all, group_kmeans,
    theta_kmeans, theta_hat, weights_kmeans,
    regime_name=f"K-means clustering (k={best_k})"
)

# ── Side-by-side welfare ratio comparison ────────────────────────────────
print("\n══ RLHF efficiency ratio W(θ̂)/W(θ̄) by regime ══")
for name, res in [("Individual", results_indiv),
                  ("Manual",     results_manual),
                  ("K-means",    results_kmeans)]:
    wdf = res["welfare_df"]
    agg = (wdf["W_rlhf"] * wdf["weight"]).sum() / (wdf["W_opt"] * wdf["weight"]).sum()
    print(f"  {name:12s}: pop-weighted rlhf_ratio = {agg:+.4f}")


In [ ]:
# ── Per-query alignment externality on synthetic test queries ─────────────
#
# Since each participant saw different queries, we construct synthetic test
# queries as α = features_A − features_B for interpretable contrasts.
# Feature ranges: size 0-4, access 0-2, income 0-5, poverty 0-7,
#                 last_donation 0-12, total_donation 0-90, dist 0-3

def make_alpha_food(diffs: dict, feat_cols=FEAT_COLS) -> np.ndarray:
    """Build α from a dict of {feature: A_val - B_val}."""
    a = np.zeros(len(feat_cols))
    for f, v in diffs.items():
        a[feat_cols.index(f)] = v
    return a

# Feature directions (higher index means):
#   size: 0=<50 clients → 4=1000 clients       (higher = larger org)
#   access: 0=normal → 2=extremely low          (higher = WORSE access)
#   income: 0=$0-20k → 5=$100k+                (higher = wealthier area)
#   poverty: 0=0-10% → 6=60+%                  (higher = more poverty, max=6)
#   last_donation: 0=never, 1=1wk ago, 12=12wks ago  (lower non-zero = more recent)
#   total_donation: 0=never → 90=12 donations   (higher = more history)
#   dist: 0=15min → 3=60+min                    (higher = farther)
test_queries = {
    # Single-feature contrasts
    "Large org vs tiny org":          make_alpha_food({"size": 4}),           # 1000 vs <50 clients
    "Good access vs extremely low":   make_alpha_food({"access": -2}),        # normal vs extremely low
    "High income vs low income area": make_alpha_food({"income": 5}),         # $100k+ vs $0-20k area
    "High poverty vs low poverty":    make_alpha_food({"poverty": 6}),        # 60+% vs 0-10%
    "Recent donor vs never donated":  make_alpha_food({"last_donation": 1}),  # 1 wk ago vs never
    "Old donor vs recent donor":      make_alpha_food({"last_donation": -11}),# 12 wks ago vs 1 wk ago
    "High vs no donation history":    make_alpha_food({"total_donation": 45}),# 45 vs 0 donations
    "Nearby vs far":                  make_alpha_food({"dist": -3}),           # 15 min vs 60+ min
    # Multi-feature compound queries
    "High poverty + good access":     make_alpha_food({"poverty": 6, "access": -2}),
    "Large org + nearby + history":   make_alpha_food({"size": 4, "dist": -3, "total_donation": 45}),
}

def query_gamma_table(test_queries, theta_groups, theta_hat, eta,
                      group_labels, regime_name):
    rows = []
    for qname, az in test_queries.items():
        adoteta    = float(az @ eta)
        sigma_hat  = float(sigmoid(az @ theta_hat))
        for g, (label, theta_g) in enumerate(zip(group_labels, theta_groups)):
            sigma_g  = float(sigmoid(az @ np.array(theta_g, dtype=float)))
            diff     = sigma_g - sigma_hat
            gamma_gz = diff * adoteta
            rows.append({
                "query":            qname,
                "group":            label,
                "σ(α·θ_g)−σ(α·θ̂)": round(diff, 4),
                "α·η":              round(adoteta, 4),
                "Γ_{g,z}":         round(gamma_gz, 4),
            })
    df = pd.DataFrame(rows)

    print(f"\n{'='*70}")
    print(f"Γ_{{g,z}} — {regime_name}")
    print(f"{'='*70}")

    pd.set_option("display.float_format", "{:+.4f}".format)
    pd.set_option("display.max_columns", None)

    print("\n── Γ_{g,z} (queries × groups) ──")
    print(df.pivot(index="query", columns="group", values="Γ_{g,z}").to_string())

    print("\n── Factor 1: σ(α·θ_g) − σ(α·θ̂) ──")
    print(df.pivot(index="query", columns="group", values="σ(α·θ_g)−σ(α·θ̂)").to_string())

    print("\n── Factor 2: α·η (query-level, same across groups) ──")
    f2 = df[df["group"] == df["group"].iloc[0]].set_index("query")[["α·η"]]
    print(f2.to_string())

    pd.reset_option("display.float_format")
    return df

# ── Regime 1: Individual ──────────────────────────────────────────────────
df_gamma_indiv = query_gamma_table(
    test_queries,
    theta_indiv, theta_hat, results_indiv["eta"],
    group_labels=person_ids,
    regime_name=f"Individual ({len(person_ids)} persons)"
)

# ── Regime 2: Manual (stakeholder type) ──────────────────────────────────
df_gamma_manual = query_gamma_table(
    test_queries,
    theta_manual, theta_hat, results_manual["eta"],
    group_labels=stakeholder_types,
    regime_name="Manual clustering (D / F / R / V)"
)

# ── Regime 3: K-means ────────────────────────────────────────────────────
df_gamma_kmeans = query_gamma_table(
    test_queries,
    theta_kmeans, theta_hat, results_kmeans["eta"],
    group_labels=[f"k{g}" for g in range(best_k)],
    regime_name=f"K-means clustering (k={best_k})"
)


In [ ]:
# ── Query contentiousness ranking over training alpha vectors ─────────────
#
# Since queries are not shared across participants, we rank the actual
# training alpha vectors (all ~N pairwise comparisons) by three metrics.
# Each alpha is labeled by its non-zero feature diffs for readability.
#
# Metrics (same as moral machine):
#   disagreement  : max_g |σ(α^T θ_g) − σ(α^T θ̂)|
#   alpha_dot_eta : |α^T η|
#   max_abs_gamma : max_g |Γ_{g,z}|  = product of the two

def label_alpha(a, feat_cols=FEAT_COLS):
    """Human-readable label: show non-zero feature diffs."""
    parts = [f"{f}:{v:+d}" for f, v in zip(feat_cols, a.astype(int)) if v != 0]
    return ", ".join(parts) if parts else "zero"

def contentiousness_ranking(alpha_queries, theta_groups, theta_hat, eta,
                             group_labels, regime_name, top_n=10):
    sig_hat   = sigmoid(alpha_queries @ theta_hat)
    sig_g_mat = np.column_stack([
        sigmoid(alpha_queries @ np.array(tg, dtype=float))
        for tg in theta_groups
    ])                                                         # (Q, K)

    adot_eta  = alpha_queries @ eta                           # (Q,)
    gamma_mat = (sig_g_mat - sig_hat[:, None]) * adot_eta[:, None]  # (Q, K)

    labels = [label_alpha(a) for a in alpha_queries]

    scores = pd.DataFrame({
        "query":          labels,
        "disagreement":   np.abs(sig_g_mat - sig_hat[:, None]).max(axis=1),
        "alpha_dot_eta":  np.abs(adot_eta),
        "max_abs_gamma":  np.abs(gamma_mat).max(axis=1),
    })

    pd.set_option("display.float_format", "{:.4f}".format)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 60)

    print(f"\n{'='*70}")
    print(f"Query contentiousness — {regime_name}")
    print(f"{'='*70}")

    for metric, label in [
        ("max_abs_gamma",  "max |Γ_{g,z}|  (welfare-consequential disagreement)"),
        ("disagreement",   "max disagreement  max_g |σ(α^T θ_g) − σ(α^T θ̂)|"),
        ("alpha_dot_eta",  "social importance  |α^T η|"),
    ]:
        print(f"\n── Top {top_n} by {label} ──")
        print(scores.nlargest(top_n, metric)[["query", "disagreement", "alpha_dot_eta", "max_abs_gamma"]]
              .to_string(index=False))
        print(f"\n── Bottom {top_n} by {label} ──")
        print(scores.nsmallest(top_n, metric)[["query", "disagreement", "alpha_dot_eta", "max_abs_gamma"]]
              .to_string(index=False))

    pd.reset_option("display.float_format")
    return scores

# ── Run for each regime ───────────────────────────────────────────────────
scores_indiv = contentiousness_ranking(
    alpha_all, theta_indiv, theta_hat, results_indiv["eta"],
    group_labels=person_ids,
    regime_name=f"Individual ({len(person_ids)} persons)"
)

scores_manual = contentiousness_ranking(
    alpha_all, theta_manual, theta_hat, results_manual["eta"],
    group_labels=stakeholder_types,
    regime_name="Manual clustering (D / F / R / V)"
)

scores_kmeans = contentiousness_ranking(
    alpha_all, theta_kmeans, theta_hat, results_kmeans["eta"],
    group_labels=[f"k{g}" for g in range(best_k)],
    regime_name=f"K-means clustering (k={best_k})"
)


## psi(theta) and Psi: geometry under the empirical query distribution

Use `alpha_all` (the actual pairwise comparison vectors from training) as the query distribution. 
Since k=7, Psi lives in R^7 -- project to 2D via PCA. 
Key question: where do the actual theta vectors (theta_hat, theta_bar per regime, per-group thetas) sit inside Psi?

In [ ]:
# from sklearn.decomposition import PCA
# from scipy.optimize import minimize

# # ── Standardize alpha vectors ─────────────────────────────────────────
# A_raw    = alpha_all.astype(float)
# feat_std = A_raw.std(axis=0)
# feat_std = np.where(feat_std < 1e-10, 1.0, feat_std)
# A_std    = A_raw / feat_std[None, :]
# k_emp    = A_std.shape[1]

# def psi_s(theta):
#     w = (sigmoid(A_std @ theta) - 0.5)
#     return (A_std * w[:, None]).mean(axis=0)

# def psi_s_batch(thetas):
#     w = (sigmoid(thetas @ A_std.T) - 0.5)
#     return (w[:, :, None] * A_std[None]).mean(axis=1)

# # ── 1. Query distribution ─────────────────────────────────────────────
# fig, axes = plt.subplots(1, 7, figsize=(20, 3))
# for ax, feat, vals in zip(axes, FEAT_COLS, A_std.T):
#     ax.hist(vals, bins=30, color='steelblue', edgecolor='white', linewidth=0.4)
#     ax.set_title(feat, fontweight='bold', fontsize=9)
#     ax.set_xlabel('standardized alpha')
#     if ax == axes[0]: ax.set_ylabel('count')
# plt.suptitle('Empirical query distribution: standardized alpha features', fontweight='bold')
# plt.tight_layout()
# plt.show()

# # ── 2. Sweep theta, project Psi to 2D via PCA ────────────────────────
# rng_psi   = np.random.default_rng(0)
# n_dirs    = 3000
# raw_dirs  = rng_psi.standard_normal((n_dirs, k_emp))
# unit_dirs = raw_dirs / np.linalg.norm(raw_dirs, axis=1, keepdims=True)
# radii     = np.concatenate([np.linspace(0, 0.5, 10),
#                              np.linspace(0.5, 5, 30),
#                              np.geomspace(5, 200, 20)])
# all_theta = np.vstack([r * unit_dirs for r in radii])
# norms_all = np.linalg.norm(all_theta, axis=1)
# all_psi   = psi_s_batch(all_theta)

# pca    = PCA(n_components=2)
# pca.fit(all_psi)
# psi_2d = pca.transform(all_psi)

# # Named theta vectors
# named_thetas = {
#     "theta_hat (RLHF)":    theta_hat,
#     "theta_bar (indiv)":   results_indiv["theta_bar_clust"],
#     "theta_bar (manual)":  results_manual["theta_bar_clust"],
#     "theta_bar (k-means)": results_kmeans["theta_bar_clust"],
# }
# for s, th in zip(stakeholder_types, theta_manual):
#     named_thetas[f"theta_{s} (manual)"] = th
# for g, th in enumerate(theta_kmeans):
#     named_thetas[f"theta_k{g} (kmeans)"] = th
# markers = ['*', 'D', 's', '^', 'o', 'P', 'X', 'v', '<', '>', 'h']

# fig, axes = plt.subplots(1, 2, figsize=(18, 7))
# for ax, mask, title in zip(
#     axes,
#     [np.ones(len(all_theta), bool), norms_all < 8],
#     ["Full Psi (PCA projection)", "Zoom: ||theta|| < 8"],
# ):
#     pts = psi_2d[mask]
#     hb = ax.hexbin(pts[:, 0], pts[:, 1], gridsize=60, cmap='YlOrRd',
#                    mincnt=1, linewidths=0)
#     plt.colorbar(hb, ax=ax, label='# theta values')
#     for (lbl, th), mk in zip(named_thetas.items(), markers):
#         p = pca.transform(psi_s(th).reshape(1, -1))
#         ax.scatter(p[0, 0], p[0, 1], s=140, marker=mk, zorder=10, label=lbl,
#                    edgecolors='black', linewidths=0.6)
#     ax.scatter([0], [0], c="black", s=60, marker="x", zorder=9)
#     ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
#     ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
#     ax.set_title(title, fontweight='bold')
#     ax.legend(fontsize=7, loc='upper left', ncol=2)
# plt.suptitle("Impact set Psi (standardized) — 412 Food Rescue", fontweight='bold', fontsize=13)
# plt.tight_layout()
# plt.show()

# print('PCA loadings:')
# print(pd.DataFrame(pca.components_.T, index=FEAT_COLS, columns=['PC1','PC2']).round(3).to_string())
# print(f'Variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}  PC2={pca.explained_variance_ratio_[1]:.1%}')

# # ── 3. Convexity test ────────────────────────────────────────────────
# def psi_inv_s(target, theta_init=None):
#     if theta_init is None: theta_init = np.zeros(k_emp)
#     def loss(th):
#         diff = psi_s(th) - target
#         return 0.5 * diff @ diff
#     def grad(th):
#         diff = psi_s(th) - target
#         v_   = sigmoid(A_std @ th) * (1 - sigmoid(A_std @ th))
#         J_   = 2 * (A_std * v_[:, None]).T @ A_std / len(A_std)
#         return J_ @ diff
#     res = minimize(loss, theta_init, jac=grad, method='L-BFGS-B',
#                    options={'maxiter': 2000, 'ftol': 1e-16})
#     return res.x, res.fun

# rng_cv   = np.random.default_rng(1)
# n_pairs  = 30
# # Sample boundary points: large-norm theta in random directions
# bdy_angles = rng_cv.standard_normal((n_pairs * 2, k_emp))
# bdy_angles /= np.linalg.norm(bdy_angles, axis=1, keepdims=True)
# bdy_pts    = psi_s_batch(500 * bdy_angles)    # points on dPsi

# residuals = []
# for i in range(n_pairs):
#     mid = (bdy_pts[2*i] + bdy_pts[2*i+1]) / 2
#     _, res = psi_inv_s(mid)
#     residuals.append(res)
# residuals = np.array(residuals)

# print(f'\nConvexity test: invert midpoints of {n_pairs} boundary point pairs')
# print(f'  Max residual: {residuals.max():.2e}  (< 1e-6 => midpoint lies in Psi => convex)')
# print(f'  Psi appears {"convex" if residuals.max() < 1e-4 else "NOT convex"} under this query distribution')


In [ ]:
# # ── 2D PCA of individual theta vectors ──────────────────────────────
# theta_matrix = theta_df[FEAT_COLS].values.astype(float)  # (N_persons, 7)
# person_ids   = theta_df.index.tolist()
# stakeholders = theta_df['stakeholder'].tolist()

# pca_th = PCA(n_components=2)
# theta_2d = pca_th.fit_transform(theta_matrix)  # (N_persons, 2)

# # Color by stakeholder type
# stype_colors = {s: c for s, c in zip(
#     sorted(set(stakeholders)),
#     ['#4878CF','#E24A33','#56A55B','#956CB4','#8C613C','#DC7EC0']
# )}
# colors = [stype_colors[s] for s in stakeholders]

# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# # Left: colored by stakeholder, labeled by personID
# ax = axes[0]
# for stype, color in stype_colors.items():
#     mask = [s == stype for s in stakeholders]
#     ax.scatter(theta_2d[mask, 0], theta_2d[mask, 1],
#                c=color, s=80, label=stype, zorder=3)
# for i, pid in enumerate(person_ids):
#     ax.annotate(pid, (theta_2d[i, 0], theta_2d[i, 1]),
#                 fontsize=7, ha='left', va='bottom',
#                 xytext=(3, 3), textcoords='offset points')
# ax.axhline(0, color='gray', lw=0.5, ls='--')
# ax.axvline(0, color='gray', lw=0.5, ls='--')
# ax.set_xlabel(f'PC1 ({pca_th.explained_variance_ratio_[0]:.1%})')
# ax.set_ylabel(f'PC2 ({pca_th.explained_variance_ratio_[1]:.1%})')
# ax.set_title('Individual thetas — colored by stakeholder', fontweight='bold')
# ax.legend(title='stakeholder', fontsize=8)

# # Right: colored by k-means cluster
# ax = axes[1]
# km_labels = theta_df['kmeans_cluster'].tolist()
# km_colors = plt.cm.tab10(np.linspace(0, 0.5, best_k))
# for g in range(best_k):
#     mask = [c == g for c in km_labels]
#     ax.scatter(theta_2d[mask, 0], theta_2d[mask, 1],
#                color=km_colors[g], s=80, label=f'cluster {g}', zorder=3)
# for i, pid in enumerate(person_ids):
#     ax.annotate(pid, (theta_2d[i, 0], theta_2d[i, 1]),
#                 fontsize=7, ha='left', va='bottom',
#                 xytext=(3, 3), textcoords='offset points')
# ax.axhline(0, color='gray', lw=0.5, ls='--')
# ax.axvline(0, color='gray', lw=0.5, ls='--')
# ax.set_xlabel(f'PC1 ({pca_th.explained_variance_ratio_[0]:.1%})')
# ax.set_ylabel(f'PC2 ({pca_th.explained_variance_ratio_[1]:.1%})')
# ax.set_title('Individual thetas — colored by k-means cluster', fontweight='bold')
# ax.legend(title='cluster', fontsize=8)

# plt.suptitle('PCA of individual theta vectors (k=7 -> 2D)', fontweight='bold', fontsize=13)
# plt.tight_layout()
# plt.show()

# print('PCA loadings (theta-space):')
# print(pd.DataFrame(pca_th.components_.T, index=FEAT_COLS,
#                    columns=['PC1','PC2']).round(3).to_string())
# print(f'Variance explained: PC1={pca_th.explained_variance_ratio_[0]:.1%}  '
#       f'PC2={pca_th.explained_variance_ratio_[1]:.1%}')


In [ ]:
# # ── Project Psi onto the theta-PCA axes ─────────────────────────────
# # all_psi and pca_th are both in R^7, so we can project psi values
# # onto the same 2D axes defined by preference variation across individuals.
# # This puts Psi and the theta points in a shared coordinate system.

# # Project swept psi values (from cell-08) using theta PCA
# psi_proj   = pca_th.transform(all_psi)        # (M, 2)  Psi in theta-PCA space
# norms_all_ = np.linalg.norm(all_theta, axis=1)

# # Project each individual's psi(theta_i)
# psi_indiv_pts = np.array([psi_s(th) for th in theta_matrix])  # (N_persons, 7)
# psi_indiv_2d  = pca_th.transform(psi_indiv_pts)               # (N_persons, 2)

# fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# for ax, mask, title in zip(
#     axes,
#     [np.ones(len(all_theta), bool), norms_all_ < 8],
#     ["Full Psi (projected to theta-PCA space)", "Zoom: ||theta|| < 8"],
# ):
#     pts = psi_proj[mask]
#     hb  = ax.hexbin(pts[:, 0], pts[:, 1], gridsize=60, cmap='YlOrRd',
#                     mincnt=1, linewidths=0, zorder=1)
#     plt.colorbar(hb, ax=ax, label='# theta values')

#     # Individual theta points
#     ax.scatter(theta_2d[:, 0], theta_2d[:, 1],
#                c=colors, s=80, zorder=4, edgecolors='black', linewidths=0.6,
#                label='theta_i')

#     # psi(theta_i) points + arrows from theta_i -> psi(theta_i)
#     ax.scatter(psi_indiv_2d[:, 0], psi_indiv_2d[:, 1],
#                c=colors, s=80, marker='x', zorder=5, linewidths=1.5,
#                label='psi(theta_i)')
#     for i in range(len(person_ids)):
#         ax.annotate('', xy=psi_indiv_2d[i], xytext=theta_2d[i],
#                     arrowprops=dict(arrowstyle='->', color='gray',
#                                    lw=0.8, alpha=0.6))

#     # Labels on theta points only
#     for i, pid in enumerate(person_ids):
#         ax.annotate(pid, theta_2d[i], fontsize=6,
#                     xytext=(3,3), textcoords='offset points')

#     ax.axhline(0, color='gray', lw=0.5, ls='--')
#     ax.axvline(0, color='gray', lw=0.5, ls='--')
#     ax.set_xlabel(f'PC1 ({pca_th.explained_variance_ratio_[0]:.1%} of theta var)')
#     ax.set_ylabel(f'PC2 ({pca_th.explained_variance_ratio_[1]:.1%} of theta var)')
#     ax.set_title(title, fontweight='bold')

# # Shared legend
# from matplotlib.lines import Line2D
# legend_els = [
#     Line2D([0],[0], marker='o', color='w', markerfacecolor='gray',
#            markeredgecolor='black', markersize=8, label='theta_i (dot)'),
#     Line2D([0],[0], marker='x', color='gray', markersize=8,
#            linewidth=1.5, label='psi(theta_i) (cross)'),
# ]
# for stype, color in stype_colors.items():
#     legend_els.append(Line2D([0],[0], marker='o', color='w',
#                              markerfacecolor=color, markersize=8, label=stype))
# axes[1].legend(handles=legend_els, fontsize=7, loc='upper left')

# plt.suptitle('Psi projected onto theta-PCA space\n'
#              'Dots = theta_i, crosses = psi(theta_i), arrows show the mapping',
#              fontweight='bold', fontsize=12)
# plt.tight_layout()
# plt.show()


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Definition 27 Distortion  (generic, works on any dataset)
#
#   Dist(µ_λ; θ) = W*(θ) / E[W(µ_λ(θ))]
#
# µ_λ = random dictatorship with weights λ (Definition 23).
# Key objects (all computed in the deterministic boundary limit):
#   θ̄_w     = Σ_n w_n θ_n                      utilitarian mean
#   ψ_n*    = E_z[α_z sign(α_z^T θ_n)]          agent n's optimal boundary impact
#   ψ_λ^sp  = Σ_n λ_n ψ_n*                      SP mechanism impact
#   W*      = E_z[|α_z^T θ̄_w|] = θ̄_w^T ψ*      utilitarian optimal welfare
#   E[W]    = θ̄_w^T ψ_λ^sp                      welfare under µ_λ
# ════════════════════════════════════════════════════════════════════

def compute_distortion(theta_agents, lambdas, alpha_queries, weights=None):
    """
    Compute Definition 27 distortion for random dictatorship µ_λ.

    Parameters
    ----------
    theta_agents  : (N, k) reported preference vectors
    lambdas       : (N,)   mechanism weights (must sum to 1)
    alpha_queries : (Q, k) deployment query vectors
    weights       : (N,)   social welfare weights w_n (default: uniform 1/N)

    Returns
    -------
    dict:
        distortion  scalar  W*(θ) / E[W(µ_λ(θ))], +inf if E[W] <= 0
        W_star      scalar  utilitarian optimal welfare
        W_sp        scalar  welfare under SP mechanism
        psi_star    (k,)    utilitarian-optimal impact vector
        psi_sp      (k,)    SP mechanism impact vector
        psi_n_star  (N, k)  each agent's boundary-optimal impact
        theta_bar   (k,)    utilitarian mean preference
    """
    theta_agents  = np.array(theta_agents,  dtype=float)
    lambdas       = np.array(lambdas,       dtype=float)
    alpha_queries = np.array(alpha_queries, dtype=float)
    N, k = theta_agents.shape

    if weights is None:
        weights = np.ones(N) / N
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()
    lambdas = lambdas / lambdas.sum()

    # 1. Utilitarian mean
    theta_bar = weights @ theta_agents                          # (k,)

    # 2. Each agent's boundary-optimal impact  ψ_n* = E_z[α_z sign(α_z^T θ_n)]
    signs_n    = np.sign(alpha_queries @ theta_agents.T)       # (Q, N)
    psi_n_star = (signs_n[:, :, None] * alpha_queries[:, None, :]).mean(axis=0)  # (N, k)

    # 3. SP impact  ψ_λ^sp = Σ_n λ_n ψ_n*
    psi_sp = lambdas @ psi_n_star                               # (k,)

    # 4. Utilitarian-optimal impact  ψ* = E_z[α_z sign(α_z^T θ̄_w)]
    psi_star = (np.sign(alpha_queries @ theta_bar)[:, None] * alpha_queries).mean(axis=0)  # (k,)

    # 5. Welfare levels
    W_star = 0.5 * float(theta_bar @ psi_star)   # = E_z[|α_z^T θ̄_w|]
    W_sp   = 0.5 * float(theta_bar @ psi_sp)

    # 6. Distortion (Definition 27)
    distortion = W_star / W_sp if W_sp > 0 else np.inf

    return dict(distortion=distortion, W_star=W_star, W_sp=W_sp,
                psi_star=psi_star, psi_sp=psi_sp,
                psi_n_star=psi_n_star, theta_bar=theta_bar)


In [ ]:
# ── Apply to 412 Food Rescue: individual-level agents ────────────────
# Switch to stakeholder or k-means by swapping theta_agents / agent_labels below.

# --- choose regime here -------------------------------------------
theta_agents = theta_df[FEAT_COLS].values.astype(float)  # (N_persons, 7)
agent_labels = theta_df.index.tolist()                   # person IDs
N_agents     = len(agent_labels)
# ------------------------------------------------------------------

A_dep = alpha_all.astype(float)   # deployment = training queries (raw scale)
w_uniform = np.ones(N_agents) / N_agents

# Three lambda choices
lambdas_uniform = np.ones(N_agents) / N_agents          # egalitarian
lambdas_welfare = w_uniform.copy()                       # lambda = w (same here)

# Welfare-maximising lambda (Eq 38): point mass on argmax_m theta_bar^T psi_m*
res_tmp   = compute_distortion(theta_agents, lambdas_uniform, A_dep, w_uniform)
n_star    = int(np.argmax(res_tmp['theta_bar'] @ res_tmp['psi_n_star'].T))
lambdas_opt = np.zeros(N_agents); lambdas_opt[n_star] = 1.0

results_dist = {}
for lbl, lam in [
    ("Uniform (egalitarian)",   lambdas_uniform),
    ("Welfare-weighted",        lambdas_welfare),
    ("Welfare-maximising (dictator)", lambdas_opt),
]:
    r = compute_distortion(theta_agents, lam, A_dep, w_uniform)
    results_dist[lbl] = r
    print(f'{lbl}:')
    print(f'  W*       = {r["W_star"]:.4f}')
    print(f'  E[W_sp]  = {r["W_sp"]:.4f}')
    print(f'  Dist     = {r["distortion"]:.4f}')
    print()

# Per-agent optimal welfare contribution  theta_bar^T psi_n*
r0 = results_dist['Uniform (egalitarian)']
agent_welfare = r0['theta_bar'] @ r0['psi_n_star'].T   # (N,)
print('Per-agent theta_bar^T psi_n* (how much utilitarian welfare each dictator achieves):')
for pid, val in sorted(zip(agent_labels, agent_welfare), key=lambda x: -x[1]):
    print(f'  {pid}: {val:.4f}')
print(f'\nWelfare-maximising dictator: {agent_labels[n_star]} '
      f'(W = {agent_welfare[n_star]:.4f}, W* = {r0["W_star"]:.4f})')


In [ ]:
# ── Distortion bootstrap & sequential addition experiments ───────────────
# Uses individual-level thetas and uniform λ throughout.
#
# Experiment 1 (bootstrap variance):
#   For each k in 2..N, draw 500 random subsets of k agents, compute
#   distortion each time → plot mean ± 1 std.
#
# Experiment 2 (sequential addition):
#   Shuffle the 19 agents into a random order, compute distortion as
#   we include the first 2, then 3, ..., then all 19.
#   Repeat 200 random orderings → plot all paths (light) + mean path.

theta_arr    = np.array(theta_indiv)                # (19, 7)
w_arr        = np.array(weights_indiv)              # (19,)  obs-fraction weights
N_agents     = len(theta_arr)
N_BOOT       = 500    # bootstrap resamples per k
N_ORDER      = 200    # random orderings for sequential experiment
A_dep        = alpha_all.astype(float)

rng = np.random.default_rng(0)

# ── Experiment 1: bootstrap variance ─────────────────────────────────────
K_vals     = list(range(2, N_agents + 1))
boot_means = []
boot_stds  = []
boot_p5    = []
boot_p95   = []

for k in K_vals:
    dists = []
    for _ in range(N_BOOT):
        idx   = rng.choice(N_agents, size=k, replace=False)
        th_k  = theta_arr[idx]
        w_k   = w_arr[idx]; w_k = w_k / w_k.sum()
        lam_k = np.ones(k) / k
        r     = compute_distortion(th_k, lam_k, A_dep, w_k)
        dists.append(r['distortion'])
    dists = np.array(dists)
    boot_means.append(dists.mean())
    boot_stds.append(dists.std())
    boot_p5.append(np.percentile(dists, 5))
    boot_p95.append(np.percentile(dists, 95))

boot_means = np.array(boot_means)
boot_stds  = np.array(boot_stds)
boot_p5    = np.array(boot_p5)
boot_p95   = np.array(boot_p95)

# ── Experiment 2: sequential addition ────────────────────────────────────
K_seq   = list(range(2, N_agents + 1))
seq_all = []   # (N_ORDER, N_agents - 1)

for _ in range(N_ORDER):
    order    = rng.permutation(N_agents)
    path     = []
    for k in K_seq:
        idx   = order[:k]
        th_k  = theta_arr[idx]
        w_k   = w_arr[idx]; w_k = w_k / w_k.sum()
        lam_k = np.ones(k) / k
        r     = compute_distortion(th_k, lam_k, A_dep, w_k)
        path.append(r['distortion'])
    seq_all.append(path)

seq_all  = np.array(seq_all)   # (N_ORDER, len(K_seq))
seq_mean = seq_all.mean(axis=0)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: bootstrap variance
ax = axes[0]
ax.fill_between(K_vals, boot_p5, boot_p95,
                alpha=0.18, color='steelblue', label='5–95th pct')
ax.fill_between(K_vals, boot_means - boot_stds, boot_means + boot_stds,
                alpha=0.35, color='steelblue', label='±1 std')
ax.plot(K_vals, boot_means, 'o-', color='steelblue', lw=2, ms=5, label='mean')
ax.axhline(1.0, color='grey', ls='--', lw=0.8)
ax.set_xlabel('Number of agents k (random subset of individuals)')
ax.set_ylabel('Distortion  W* / E[W_sp]')
ax.set_title('Experiment 1: bootstrap variance' + f' ({N_BOOT} samples per k, uniform λ)',
             fontweight='bold')
ax.set_xticks(K_vals)
ax.legend(fontsize=8)

# Right: sequential addition
ax = axes[1]
for path in seq_all:
    ax.plot(K_seq, path, color='steelblue', alpha=0.06, lw=1)
ax.plot(K_seq, seq_mean, color='steelblue', lw=2.5, label='mean across orderings')
ax.axhline(1.0, color='grey', ls='--', lw=0.8)
ax.set_xlabel('Number of agents k (agents added one at a time)')
ax.set_ylabel('Distortion  W* / E[W_sp]')
ax.set_title('Experiment 2: sequential addition' + f' ({N_ORDER} random orderings, uniform λ)',
             fontweight='bold')
ax.set_xticks(K_seq)
ax.legend(fontsize=8)

plt.suptitle('Distortion as a function of agent count — 412 Food Rescue (n=19 individuals)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Bootstrap: distortion at k=2: {boot_means[0]:.3f} ± {boot_stds[0]:.3f}")
print(f"Bootstrap: distortion at k=19 (all): {boot_means[-1]:.3f} ± {boot_stds[-1]:.4f}")
print(f"Sequential mean path range: [{seq_mean.min():.3f}, {seq_mean.max():.3f}]")


In [ ]:
# ── Social welfare vs sigmoid temperature β ───────────────────────────────
#
# W(θ_c; β) = Σ_n w_n · E_z[ 2(α·θ_n)(σ(β·α·θ_c) − 0.5) ]
# Normalized by W* = E_z[|α·θ̄|].
# Three mechanisms: utilitarian θ̄, RLHF θ̂, strategyproof θ_sp(β).

from scipy.special import expit
from scipy.optimize import minimize

theta_arr  = theta_df[FEAT_COLS].values.astype(float)
w_uniform  = np.ones(len(theta_arr)) / len(theta_arr)
theta_bar  = w_uniform @ theta_arr
theta_rlhf = theta_hat

def phi(theta, beta, alpha):
    return ((expit(beta * (alpha @ theta)) - 0.5)[:, None] * alpha).mean(axis=0)

def phi_inv(target, beta, alpha, theta_init=None):
    if theta_init is None:
        theta_init = np.zeros(alpha.shape[1])
    def loss_and_grad(theta):
        s        = expit(beta * (alpha @ theta))
        phi_t    = ((s - 0.5)[:, None] * alpha).mean(axis=0)
        residual = phi_t - target
        v        = s * (1 - s)
        J        = (alpha * (beta * v)[:, None]).T @ alpha / len(alpha)
        return 0.5 * np.dot(residual, residual), J @ residual
    res = minimize(loss_and_grad, theta_init, jac=True, method='L-BFGS-B',
                   options={'maxiter': 500, 'ftol': 1e-14, 'gtol': 1e-8})
    return res.x

def social_welfare_beta(theta_deploy, beta, theta_agents, weights, alpha):
    factor = (expit(beta * (alpha @ theta_deploy)) - 0.5)
    return float(weights @ (alpha @ theta_agents.T * factor[:, None]).mean(axis=0))

psi_star = (np.sign(alpha_all @ theta_bar)[:, None] * alpha_all).mean(axis=0)
W_star   = 0.5 * float(theta_bar @ psi_star)

betas = np.logspace(-1, 1.5, 100)

W_util_sw, W_rlhf_sw, W_sp_sw = [], [], []
theta_sp_prev = theta_bar.copy()

for b in betas:
    phi_avg       = np.stack([phi(th, b, alpha_all) for th in theta_arr]).mean(axis=0)
    theta_sp      = phi_inv(phi_avg, b, alpha_all, theta_init=theta_sp_prev)
    theta_sp_prev = theta_sp.copy()

    W_util_sw.append(social_welfare_beta(theta_bar,  b, theta_arr, w_uniform, alpha_all))
    W_rlhf_sw.append(social_welfare_beta(theta_rlhf, b, theta_arr, w_uniform, alpha_all))
    W_sp_sw.append(  social_welfare_beta(theta_sp,   b, theta_arr, w_uniform, alpha_all))

W_util_sw = np.array(W_util_sw) / W_star
W_rlhf_sw = np.array(W_rlhf_sw) / W_star
W_sp_sw   = np.array(W_sp_sw)   / W_star

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(betas, W_util_sw, color='#2166ac', ls='-',  lw=2.2, label=r'Utilitarian $\bar\theta$')
ax.plot(betas, W_rlhf_sw, color='#d73027', ls='--', lw=2.2, label=r'RLHF $\hat\theta$')
ax.plot(betas, W_sp_sw,   color='#f4a736', ls='-',  lw=2.2, label=r'Strategyproof $\theta_{sp}(\beta)$')
ax.axhline(1.0, color='grey', ls='--', lw=0.9, alpha=0.7,
           label=r'$W^*$ (ideal limit, $\beta\to\infty$)')
ax.set_xscale('log')
ax.set_xlabel(r'Sigmoid temperature $\beta$', fontsize=12)
ax.set_ylabel(r'Social welfare  $W(\theta_c;\,\beta)\;/\;W^*$', fontsize=12)
ax.set_title('412 Food Rescue: social welfare vs sigmoid temperature β\n'
             'three mechanisms  (individual-level agents, uniform weights)',
             fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, which='both', alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# ── Per-person welfare bounds vs sigmoid temperature β ────────────────────
#
# For each mechanism and each β, compute W_n(θ_c(β); β) for all N agents,
# then take the pointwise max and min across agents.
# The "best-off" and "worst-off" person can switch at each β value.
#
# Three subplots: one per mechanism (utilitarian, RLHF, strategyproof).
# Shaded band = range [min, max]; dashed line = social welfare (mean).

from scipy.special import expit
from scipy.optimize import minimize

theta_arr  = theta_df[FEAT_COLS].values.astype(float)
w_uniform  = np.ones(len(theta_arr)) / len(theta_arr)
theta_bar  = w_uniform @ theta_arr
theta_rlhf = theta_hat

def phi(theta, beta, alpha):
    return ((expit(beta * (alpha @ theta)) - 0.5)[:, None] * alpha).mean(axis=0)

def phi_inv(target, beta, alpha, theta_init=None):
    if theta_init is None:
        theta_init = np.zeros(alpha.shape[1])
    def loss_and_grad(theta):
        s        = expit(beta * (alpha @ theta))
        phi_t    = ((s - 0.5)[:, None] * alpha).mean(axis=0)
        residual = phi_t - target
        v        = s * (1 - s)
        J        = (alpha * (beta * v)[:, None]).T @ alpha / len(alpha)
        return 0.5 * np.dot(residual, residual), J @ residual
    res = minimize(loss_and_grad, theta_init, jac=True, method='L-BFGS-B',
                   options={'maxiter': 500, 'ftol': 1e-14, 'gtol': 1e-8})
    return res.x

def per_agent_welfare(theta_deploy, beta, theta_agents, alpha):
    factor = (expit(beta * (alpha @ theta_deploy)) - 0.5)
    return (alpha @ theta_agents.T * factor[:, None]).mean(axis=0)  # (N,)

psi_star = (np.sign(alpha_all @ theta_bar)[:, None] * alpha_all).mean(axis=0)
W_star   = 0.5 * float(theta_bar @ psi_star)

betas = np.logspace(-1, 1.5, 100)

records = {lbl: {'max': [], 'min': [], 'mean': []}
           for lbl in ['Utilitarian', 'RLHF', 'Strategyproof']}

theta_sp_prev = theta_bar.copy()

for b in betas:
    phi_avg       = np.stack([phi(th, b, alpha_all) for th in theta_arr]).mean(axis=0)
    theta_sp      = phi_inv(phi_avg, b, alpha_all, theta_init=theta_sp_prev)
    theta_sp_prev = theta_sp.copy()

    for lbl, th_c in [('Utilitarian', theta_bar),
                      ('RLHF',        theta_rlhf),
                      ('Strategyproof', theta_sp)]:
        W_n = per_agent_welfare(th_c, b, theta_arr, alpha_all) / W_star
        records[lbl]['max'].append(W_n.max())
        records[lbl]['min'].append(W_n.min())
        records[lbl]['mean'].append(W_n.mean())

for lbl in records:
    for k in records[lbl]:
        records[lbl][k] = np.array(records[lbl][k])

# ── Plot ──────────────────────────────────────────────────────────────────
mech_colors = {
    'Utilitarian':   '#2166ac',
    'RLHF':          '#d73027',
    'Strategyproof': '#f4a736',
}
mech_titles = {
    'Utilitarian':   r'Utilitarian  $\bar\theta$',
    'RLHF':          r'RLHF  $\hat\theta$',
    'Strategyproof': r'Strategyproof  $\theta_{sp}(\beta)$',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, lbl in zip(axes, ['Utilitarian', 'RLHF', 'Strategyproof']):
    c   = mech_colors[lbl]
    rec = records[lbl]

    ax.fill_between(betas, rec['min'], rec['max'],
                    color=c, alpha=0.18, label='worst-off to best-off range')
    ax.plot(betas, rec['max'],  color=c, lw=2.0, ls='-',  label='best-off person')
    ax.plot(betas, rec['min'],  color=c, lw=2.0, ls=':',  label='worst-off person')
    ax.plot(betas, rec['mean'], color=c, lw=1.5, ls='--', label='social welfare (mean)')
    ax.axhline(0, color='grey', lw=0.7, alpha=0.5)

    ax.set_xscale('log')
    ax.set_xlabel(r'Sigmoid temperature $\beta$', fontsize=11)
    ax.set_title(mech_titles[lbl], fontweight='bold', fontsize=11)
    ax.grid(True, which='both', alpha=0.18)
    ax.legend(fontsize=8, loc='lower right')

axes[0].set_ylabel(r'Individual welfare  $W_n(\theta_c;\,\beta)\;/\;W^*$', fontsize=11)
fig.suptitle('412 Food Rescue: best-off and worst-off individual welfare vs β\n'
             '(person identity can switch at each β; shaded = full range across agents)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ── Agent identity diagnostic: which agent is best/worst-off at each β ───
from scipy.special import expit
from scipy.optimize import minimize
from collections import Counter

theta_arr  = theta_df[FEAT_COLS].values.astype(float)
w_uniform  = np.ones(len(theta_arr)) / len(theta_arr)
theta_bar  = w_uniform @ theta_arr
theta_rlhf = theta_hat
agent_ids  = list(theta_df.index)   # e.g. ['D1', 'D2', ..., 'V2']
N          = len(theta_arr)

def _phi(theta, beta, alpha):
    return ((expit(beta * (alpha @ theta)) - 0.5)[:, None] * alpha).mean(axis=0)

def _phi_inv(target, beta, alpha, theta_init=None):
    if theta_init is None:
        theta_init = np.zeros(alpha.shape[1])
    def loss_and_grad(theta):
        s        = expit(beta * (alpha @ theta))
        phi_t    = ((s - 0.5)[:, None] * alpha).mean(axis=0)
        residual = phi_t - target
        v        = s * (1 - s)
        J        = (alpha * (beta * v)[:, None]).T @ alpha / len(alpha)
        return 0.5 * np.dot(residual, residual), J @ residual
    res = minimize(loss_and_grad, theta_init, jac=True, method='L-BFGS-B',
                   options={'maxiter': 500, 'ftol': 1e-14, 'gtol': 1e-8})
    return res.x

def _per_agent_welfare(theta_deploy, beta, theta_agents, alpha):
    factor = (expit(beta * (alpha @ theta_deploy)) - 0.5)
    return (alpha @ theta_agents.T * factor[:, None]).mean(axis=0)

psi_star = (np.sign(alpha_all @ theta_bar)[:, None] * alpha_all).mean(axis=0)
W_star   = 0.5 * float(theta_bar @ psi_star)

betas = np.logspace(-1, 1.5, 100)

argrecords = {lbl: {'argmin': [], 'argmax': []}
              for lbl in ['Utilitarian', 'RLHF', 'Strategyproof']}

theta_sp_prev = theta_bar.copy()
for b in betas:
    phi_avg       = np.stack([_phi(th, b, alpha_all) for th in theta_arr]).mean(axis=0)
    theta_sp      = _phi_inv(phi_avg, b, alpha_all, theta_init=theta_sp_prev)
    theta_sp_prev = theta_sp.copy()
    for lbl, th_c in [('Utilitarian',   theta_bar),
                      ('RLHF',          theta_rlhf),
                      ('Strategyproof', theta_sp)]:
        W_n = _per_agent_welfare(th_c, b, theta_arr, alpha_all) / W_star
        argrecords[lbl]['argmin'].append(int(W_n.argmin()))
        argrecords[lbl]['argmax'].append(int(W_n.argmax()))

# ── Plot ──────────────────────────────────────────────────────────────────
SK_COLORS  = {'D': '#e07b54', 'F': '#5b8db8', 'R': '#6abf69', 'V': '#9b6ebf'}
SK_LABELS  = {'D': 'Donor', 'F': 'Food bank', 'R': 'Recipient', 'V': 'Volunteer'}
mech_titles = {
    'Utilitarian':   r'Utilitarian  $\bar\theta$',
    'RLHF':          r'RLHF  $\hat\theta$',
    'Strategyproof': r'Strategyproof  $\theta_{sp}(\beta)$',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, lbl in zip(axes, ['Utilitarian', 'RLHF', 'Strategyproof']):
    amin_idx = argrecords[lbl]['argmin']
    amax_idx = argrecords[lbl]['argmax']

    for bi, (b, ai) in enumerate(zip(betas, amin_idx)):
        sk = agent_ids[ai][0]
        ax.scatter(b, ai, color=SK_COLORS[sk], s=22, alpha=0.85, marker='v', zorder=3)

    for bi, (b, ai) in enumerate(zip(betas, amax_idx)):
        sk = agent_ids[ai][0]
        ax.scatter(b, ai, color=SK_COLORS[sk], s=22, alpha=0.85, marker='^', zorder=3)

    ax.set_xscale('log')
    ax.set_xlabel(r'$\beta$', fontsize=11)
    ax.set_yticks(range(N))
    ax.set_yticklabels(agent_ids, fontsize=8)
    ax.set_title(mech_titles[lbl], fontweight='bold', fontsize=11)
    ax.grid(True, which='both', alpha=0.15)

axes[0].set_ylabel('Agent', fontsize=11)

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
sk_handles  = [Patch(facecolor=SK_COLORS[k], label=f'{k} — {SK_LABELS[k]}')
               for k in sorted(SK_COLORS)]
role_handles = [Line2D([0],[0], marker='^', color='grey', ls='none', ms=8, label='best-off'),
                Line2D([0],[0], marker='v', color='grey', ls='none', ms=8, label='worst-off')]
axes[2].legend(handles=sk_handles + role_handles, fontsize=8, loc='upper right')

fig.suptitle('412 Food Rescue: which agent is best-off / worst-off at each β\n'
             '(▲ = best-off, ▼ = worst-off; color = stakeholder type)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print("Worst-off agent (argmin) across beta values:")
for lbl in ['Utilitarian', 'RLHF', 'Strategyproof']:
    ids = [agent_ids[i] for i in argrecords[lbl]['argmin']]
    cnt = Counter(ids)
    print(f"  {lbl:14s}: {dict(sorted(cnt.items(), key=lambda x: -x[1]))}")

print("\nBest-off agent (argmax) across beta values:")
for lbl in ['Utilitarian', 'RLHF', 'Strategyproof']:
    ids = [agent_ids[i] for i in argrecords[lbl]['argmax']]
    cnt = Counter(ids)
    print(f"  {lbl:14s}: {dict(sorted(cnt.items(), key=lambda x: -x[1]))}")


In [ ]:
# ── Diagnostic: per-agent welfare values and alpha distribution ───────────
#
# Checks whether any agent has negative welfare and examines whether
# the query distribution alpha is skewed (biased away from zero),
# which would suppress negative welfare even for opposing-theta agents.

from scipy.special import expit

theta_arr  = theta_df[FEAT_COLS].values.astype(float)
w_uniform  = np.ones(len(theta_arr)) / len(theta_arr)
theta_bar  = w_uniform @ theta_arr
theta_rlhf = theta_hat

def per_agent_welfare(theta_deploy, beta, theta_agents, alpha):
    factor = (expit(beta * (alpha @ theta_deploy)) - 0.5)
    return (alpha @ theta_agents.T * factor[:, None]).mean(axis=0)

# ── 1. Mean and std of alpha vectors per feature ──────────────────────────
print("── Alpha vector distribution (features_A − features_B) ──")
alpha_df = pd.DataFrame(alpha_all, columns=FEAT_COLS)
print(alpha_df.describe().loc[['mean', 'std', 'min', 'max']].round(3).to_string())
print(f"\nFraction of alpha vectors that are all-zero: "
      f"{(np.abs(alpha_all).sum(axis=1) == 0).mean():.3f}")

# ── 2. Per-agent welfare at several beta values ───────────────────────────
print("\n── Per-agent welfare W_n(θ̄; β) for utilitarian deployment ──")
print(f"{'Person':>10}", end="")
check_betas = [0.5, 1.0, 5.0, 10.0]
for b in check_betas:
    print(f"  β={b:4.1f}", end="")
print()
for b in check_betas:
    W_n = per_agent_welfare(theta_bar, b, theta_arr, alpha_all)
    if b == check_betas[0]:
        for pid, w in zip(theta_df.index, W_n):
            print(f"{pid:>10}", end="")
            print(f"  {w:+.4f}", end="")
    else:
        W_n_vals = per_agent_welfare(theta_bar, b, theta_arr, alpha_all)

# Recompute cleanly as a table
rows = []
for b in check_betas:
    W_n = per_agent_welfare(theta_bar, b, theta_arr, alpha_all)
    for pid, w in zip(theta_df.index, W_n):
        rows.append({'person': pid, 'beta': b, 'W_util': w})
welfare_table = pd.DataFrame(rows).pivot(index='person', columns='beta', values='W_util')
welfare_table.columns = [f'β={b}' for b in check_betas]
print(welfare_table.round(4).to_string())

print(f"\nAny negative welfare under utilitarian θ̄: "
      f"{(welfare_table.values < 0).any()}")

print("\n── Per-agent welfare W_n(θ̂; β=5) for RLHF deployment ──")
W_rlhf_5 = per_agent_welfare(theta_rlhf, 5.0, theta_arr, alpha_all)
for pid, w in zip(theta_df.index, W_rlhf_5):
    print(f"  {pid}: {w:+.4f}")
print(f"\nAny negative welfare under RLHF θ̂ at β=5: {(W_rlhf_5 < 0).any()}")

# ── 3. Signed alpha dot theta_n for agents with opposing features ─────────
print("\n── E_z[α^T θ_n] per agent (unsigned overlap with query distribution) ──")
mean_alpha = alpha_all.mean(axis=0)
for pid, th in zip(theta_df.index, theta_arr):
    dot = float(mean_alpha @ th)
    print(f"  {pid}: E[α]·θ_n = {dot:+.4f}")


In [ ]:
# ── Gini coefficient of per-agent welfare vs sigmoid temperature β ────────
#
# Strategyproof theta (at each β) is defined via the impact function:
#   Φ(θ; β) = E_z[ 2(σ(β·α^T θ) − 0.5) · α ]
#
#   θ_sp(β) = Φ⁻¹( (1/N) Σ_i Φ(θ_i; β) )
#
# i.e. find the single theta whose impact vector equals the uniform convex
# combination of all individual impact vectors. Solved numerically via L-BFGS-B.
#
# Then per-agent welfare under each mechanism at temperature β:
#   Utilitarian:    W_n(θ̄;    β)
#   RLHF:           W_n(θ̂;    β)
#   Strategyproof:  W_n(θ_sp; β)
#
# Gini coefficient computed over the N-agent welfare vector.
# Values are shifted to non-negative before applying the standard formula.

from scipy.special import expit
from scipy.optimize import minimize

theta_arr  = theta_df[FEAT_COLS].values.astype(float)
w_uniform  = np.ones(len(theta_arr)) / len(theta_arr)
theta_bar  = w_uniform @ theta_arr
theta_rlhf = theta_hat

def phi(theta, beta, alpha):
    """Impact vector Φ(θ; β) = E_z[ 2(σ(β·α^T θ) − 0.5) · α ]"""
    return ((expit(beta * (alpha @ theta)) - 0.5)[:, None] * alpha).mean(axis=0)

def phi_inv(target, beta, alpha, theta_init=None):
    """Find θ s.t. Φ(θ; β) ≈ target via L-BFGS-B with analytic gradient."""
    if theta_init is None:
        theta_init = np.zeros(alpha.shape[1])
    def loss_and_grad(theta):
        s        = expit(beta * (alpha @ theta))                      # (Q,)
        phi_t    = ((s - 0.5)[:, None] * alpha).mean(axis=0)     # (k,)
        residual = phi_t - target                                     # (k,)
        loss     = 0.5 * np.dot(residual, residual)
        v        = s * (1 - s)                                        # (Q,)
        J        = (alpha * (beta * v)[:, None]).T @ alpha / len(alpha)  # (k, k)
        return loss, J @ residual
    res = minimize(loss_and_grad, theta_init, jac=True, method='L-BFGS-B',
                   options={'maxiter': 500, 'ftol': 1e-14, 'gtol': 1e-8})
    return res.x

def per_agent_welfare(theta_deploy, beta, theta_agents, alpha):
    sigma_beta = expit(beta * (alpha @ theta_deploy))
    factor     = (sigma_beta - 0.5)
    return (alpha @ theta_agents.T * factor[:, None]).mean(axis=0)  # (N,)

def gini(values):
    v = np.array(values, dtype=float)
    v = v - v.min()
    if v.sum() < 1e-12:
        return 0.0
    v = np.sort(v)
    N = len(v)
    return float((2 * (np.arange(1, N + 1) * v).sum()) / (N * v.sum()) - (N + 1) / N)

betas = np.logspace(-1, 1.5, 100)  # β ∈ [0.1, ~32]

gini_util = []
gini_rlhf = []
gini_sp   = []

theta_sp_prev = theta_bar.copy()   # warm-start the inversion

for b in betas:
    # (1/N) Σ_i Φ(θ_i; β)  — target impact vector for SP
    phi_avg = np.stack([phi(th, b, alpha_all) for th in theta_arr]).mean(axis=0)

    # θ_sp(β) = Φ⁻¹(phi_avg)
    theta_sp      = phi_inv(phi_avg, b, alpha_all, theta_init=theta_sp_prev)
    theta_sp_prev = theta_sp.copy()

    W_util = per_agent_welfare(theta_bar,  b, theta_arr, alpha_all)
    W_rlhf = per_agent_welfare(theta_rlhf, b, theta_arr, alpha_all)
    W_sp   = per_agent_welfare(theta_sp,   b, theta_arr, alpha_all)

    gini_util.append(gini(W_util))
    gini_rlhf.append(gini(W_rlhf))
    gini_sp.append(gini(W_sp))

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(betas, gini_util, color='#2166ac', lw=2.2, ls='-',
        label=r'Utilitarian $\bar\theta$')
ax.plot(betas, gini_rlhf, color='#d73027', lw=2.2, ls='--',
        label=r'RLHF $\hat\theta$')
ax.plot(betas, gini_sp,   color='#f4a736', lw=2.2, ls='-.',
        label=r'Strategyproof $\theta_{sp}(\beta)$')

ax.set_xscale('log')
ax.set_ylim(bottom=0)
ax.set_xlabel(r'Sigmoid temperature $\beta$', fontsize=12)
ax.set_ylabel('Gini coefficient of per-agent welfare', fontsize=12)
ax.set_title('412 Food Rescue: welfare inequality across agents\n'
             r'Gini coefficient vs sigmoid temperature $\beta$',
             fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.2)
plt.tight_layout()
plt.show()

for lbl, arr in [('Utilitarian', gini_util), ('RLHF', gini_rlhf), ('SP', gini_sp)]:
    print(f'{lbl:14s}  β=0.1: {arr[0]:.3f}   β=1: {arr[np.argmin(np.abs(betas-1))]:.3f}   β=32: {arr[-1]:.3f}')


In [ ]:
# ── Welfare variance across agents vs β — 412 Food Rescue ────────────────
#
# For each mechanism and each β:
#   Var[W_n(θ_c; β)] = (1/N) Σ_n (W_n − W̄)²
# Normalized by W*² to give a dimensionless relative variance.

from scipy.special import expit
from scipy.optimize import minimize

theta_arr  = theta_df[FEAT_COLS].values.astype(float)
w_uniform  = np.ones(len(theta_arr)) / len(theta_arr)
theta_bar  = w_uniform @ theta_arr
theta_rlhf = theta_hat

def phi_all(theta_agents, beta, alpha):
    logits = alpha @ theta_agents.T
    factor = (expit(beta * logits) - 0.5)
    return factor.T @ alpha / len(alpha)

def phi_inv(target, beta, alpha, theta_init=None):
    if theta_init is None:
        theta_init = np.zeros(alpha.shape[1])
    def loss_and_grad(theta):
        s        = expit(beta * (alpha @ theta))
        phi_t    = ((s - 0.5)[:, None] * alpha).mean(axis=0)
        residual = phi_t - target
        v        = s * (1 - s)
        J        = (alpha * (beta * v)[:, None]).T @ alpha / len(alpha)
        return 0.5 * np.dot(residual, residual), J @ residual
    res = minimize(loss_and_grad, theta_init, jac=True, method='L-BFGS-B',
                   options={'maxiter': 500, 'ftol': 1e-14, 'gtol': 1e-8})
    return res.x

def per_agent_welfare(theta_deploy, beta, theta_agents, alpha):
    factor = (expit(beta * (alpha @ theta_deploy)) - 0.5)
    return (alpha @ theta_agents.T * factor[:, None]).mean(axis=0)

psi_star  = (np.sign(alpha_all @ theta_bar)[:, None] * alpha_all).mean(axis=0)
W_star    = 0.5 * float(theta_bar @ psi_star)
betas     = np.logspace(-1, 1.5, 100)

var_util, var_rlhf, var_sp = [], [], []
theta_sp_prev = theta_bar.copy()

for b in betas:
    phi_avg       = phi_all(theta_arr, b, alpha_all).mean(axis=0)
    theta_sp      = phi_inv(phi_avg, b, alpha_all, theta_init=theta_sp_prev)
    theta_sp_prev = theta_sp.copy()

    W_util = per_agent_welfare(theta_bar,  b, theta_arr, alpha_all)
    W_rlhf = per_agent_welfare(theta_rlhf, b, theta_arr, alpha_all)
    W_sp   = per_agent_welfare(theta_sp,   b, theta_arr, alpha_all)

    W_star_sq = W_star ** 2
    var_util.append(W_util.var() / W_star_sq)
    var_rlhf.append(W_rlhf.var() / W_star_sq)
    var_sp.append(  W_sp.var()   / W_star_sq)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(betas, var_util, color='#2166ac', lw=2.2, ls='-',
        label=r'Utilitarian $\bar\theta$')
ax.plot(betas, var_rlhf, color='#d73027', lw=2.2, ls='--',
        label=r'RLHF $\hat\theta$')
ax.plot(betas, var_sp,   color='#f4a736', lw=2.2, ls='-.',
        label=r'Strategyproof $\theta_{sp}(\beta)$')
ax.set_xscale('log')
ax.set_ylim(bottom=0)
ax.set_xlabel(r'Sigmoid temperature $\beta$', fontsize=12)
ax.set_ylabel(r'Var$[W_n / W^*]$', fontsize=12)
ax.set_title('412 Food Rescue: welfare variance across 19 agents\n'
             r'vs sigmoid temperature $\beta$', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.2)
plt.tight_layout()
plt.show()

for lbl, arr in [('Utilitarian', var_util), ('RLHF', var_rlhf), ('SP', var_sp)]:
    print(f'{lbl:14s}  β=0.1: {arr[0]:.4f}   β=1: {arr[np.argmin(np.abs(betas-1))]:.4f}'
          f'   β=32: {arr[-1]:.4f}')
